In [1]:
# Data manipulation and numerical computation
import numpy as np
import pandas as pd
import os

# pandas all columns display
pd.set_option('display.max_columns', None)

# Load data

In [2]:
path = "../../data/processed/"
dfs = {}

# read 03_CLEAN_COMPLETE_DF.parquet
df1 = pd.read_parquet(os.path.join(path, "taxones_pressure_epm_train.parquet"))
df2 = pd.read_parquet(os.path.join(path, "taxones_pressure_epm_predict.parquet"))
df = pd.concat([df1, df2], ignore_index=True)

In [3]:
# change unassessed values to NaN
df = df.replace("Unassessed", np.nan)
df = df.replace("None", np.nan)

# index SamplingOperations_code
df = df.set_index('SamplingOperations_code')


# sort by Date_SamplingOperation
df = df.sort_values(by='Date_SamplingOperation')

df = df[['CodeSite_SamplingOperations_x', 'Date_SamplingOperation', 'IBD', 'IBD_EQR', 'IBD_EQR_Status', 'Longitude_Lambert93', 'Latitude_Lambert93', 'Watershed', 'HERlvl1Name', 'Streamsize']]
# DROP HERlvl2Code	Altitude Longitude_Lambert93	Latitude_Lambert93	Watershed	CodeDepartement	HERlvl1Code
# df = df.drop(columns=['HERlvl2Code', 'HERlvl2Name', 'HERlvl1Name', 'Altitude','Longitude_Lambert93','Latitude_Lambert93','Watershed','CodeDepartement', 'Date_SamplingOperation'])
df

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02115200_20070702,S02115200,2007-07-02,10.5,0.555556,Moderate,902925.0,6941112.0,Rhin-Meuse,COTES CALCAIRES EST,M
S05218450_20070709,S05218450,2007-07-09,20.0,1.000000,High,442885.0,6199880.0,Adour-Garonne,PYRENEES,TP
S05217350_20070709,S05217350,2007-07-09,20.0,1.000000,High,442250.0,6218040.0,Adour-Garonne,PYRENEES,TP
S05235850_20070709,S05235850,2007-07-09,NaN,NaN,NaN,463192.0,6225280.0,Adour-Garonne,COTEAUX AQUITAINS,TP
S05234290_20070709,S05234290,2007-07-09,16.6,0.773333,Moderate,456573.0,6224410.0,Adour-Garonne,PYRENEES,TP
...,...,...,...,...,...,...,...,...,...,...
S06580577_20230907,S06580577,2023-09-07,14.5,0.633333,Moderate,908596.0,6499943.0,Rhône-Méditerranée,JURA-PREALPES DU NORD,TP
S06001284_20230907,S06001284,2023-09-07,16.3,0.753333,Moderate,910629.0,6500149.0,Rhône-Méditerranée,JURA-PREALPES DU NORD,TP
S06001533_20230907,S06001533,2023-09-07,14.8,0.653333,Moderate,909322.0,6499820.0,Rhône-Méditerranée,JURA-PREALPES DU NORD,TP


In [4]:
import numpy as np
from scipy.spatial import distance

def get_nans(df):
    nans_ibd = df['IBD'].isna().sum()
    nans_ibd_eqr = df['IBD_EQR'].isna().sum()
    nans_status = df['IBD_EQR_Status'].isna().sum()
    print(f"Total NaNs in IBD: {nans_ibd}")
    print(f"Total NaNs in IBD_EQR: {nans_ibd_eqr}")
    print(f"Total NaNs in IBD_EQR_Status: {nans_status}")

# CodeSite Sampling Operatiens

In [5]:
# how many different CodeSite_SamplingOperations_x are there?
df['CodeSite_SamplingOperations_x'].nunique()

8404

In [6]:
# create a dictionary of dataframes per site and later access them by 1, 2, ...
sites = {}
for site, group in df.groupby('CodeSite_SamplingOperations_x'):
    sites[site] = group

In [7]:
site = sites['S05151150']
site

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S05151150_20070809,S05151150,2007-08-09,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20080704,S05151150,2008-07-04,19.8,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20090707,S05151150,2009-07-07,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20100804,S05151150,2010-08-04,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20110614,S05151150,2011-06-14,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20110719,S05151150,2011-07-19,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20120717,S05151150,2012-07-17,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20120806,S05151150,2012-08-06,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20130708,S05151150,2013-07-08,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP


In [8]:
def interpolate(site):
    # make interpolation of IBD and the IBD_EQR columns
    site['IBD'] = site['IBD'].interpolate(method='linear')
    site['IBD_EQR'] = site['IBD_EQR'].interpolate(method='linear')
    return site

def safe_fill(site, col="IBD_EQR_Status", date_col="Date_SamplingOperation"):
    site = site.sort_values(date_col).copy()
    prev, nxt = site[col].ffill(), site[col].bfill()
    mask = site[col].isna() & prev.eq(nxt) & prev.notna()
    site.loc[mask, col] = prev[mask]
    return site

## ejemplo

In [9]:
site = safe_fill(site)
site

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S05151150_20070809,S05151150,2007-08-09,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20080704,S05151150,2008-07-04,19.8,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20090707,S05151150,2009-07-07,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20100804,S05151150,2010-08-04,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20110614,S05151150,2011-06-14,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20110719,S05151150,2011-07-19,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20120717,S05151150,2012-07-17,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20120806,S05151150,2012-08-06,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20130708,S05151150,2013-07-08,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP


In [10]:
site = interpolate(site)
site

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S05151150_20070809,S05151150,2007-08-09,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20080704,S05151150,2008-07-04,19.8,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20090707,S05151150,2009-07-07,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20100804,S05151150,2010-08-04,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20110614,S05151150,2011-06-14,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20110719,S05151150,2011-07-19,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20120717,S05151150,2012-07-17,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20120806,S05151150,2012-08-06,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP
S05151150_20130708,S05151150,2013-07-08,20.0,1.0,High,760560.0,6352370.0,Adour-Garonne,CEVENNES,TP


## otro sitio

In [11]:
site = sites['S02061500']
site

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02061500_20070806,S02061500,2007-08-06,13.8,0.628571,Moderate,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20080721,S02061500,2008-07-21,11.3,0.450000,Poor,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20090820,S02061500,2009-08-20,NaN,NaN,NaN,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20100721,S02061500,2010-07-21,16.1,0.792857,Good,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20110704,S02061500,2011-07-04,15.7,0.764286,Moderate,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20120716,S02061500,2012-07-16,NaN,NaN,NaN,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20130718,S02061500,2013-07-18,16.6,0.828571,Good,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20140915,S02061500,2014-09-15,16.3,0.807143,Good,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20140923,S02061500,2014-09-23,12.8,0.557143,Moderate,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP


In [12]:
safe_fill(site)

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02061500_20070806,S02061500,2007-08-06,13.8,0.628571,Moderate,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20080721,S02061500,2008-07-21,11.3,0.450000,Poor,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20090820,S02061500,2009-08-20,NaN,NaN,NaN,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20100721,S02061500,2010-07-21,16.1,0.792857,Good,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20110704,S02061500,2011-07-04,15.7,0.764286,Moderate,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20120716,S02061500,2012-07-16,NaN,NaN,NaN,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20130718,S02061500,2013-07-18,16.6,0.828571,Good,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20140915,S02061500,2014-09-15,16.3,0.807143,Good,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20140923,S02061500,2014-09-23,12.8,0.557143,Moderate,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP


In [13]:
interpolate(site)

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02061500_20070806,S02061500,2007-08-06,13.80,0.628571,Moderate,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20080721,S02061500,2008-07-21,11.30,0.450000,Poor,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20090820,S02061500,2009-08-20,13.70,0.621429,NaN,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20100721,S02061500,2010-07-21,16.10,0.792857,Good,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20110704,S02061500,2011-07-04,15.70,0.764286,Moderate,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20120716,S02061500,2012-07-16,16.15,0.796429,NaN,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20130718,S02061500,2013-07-18,16.60,0.828571,Good,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20140915,S02061500,2014-09-15,16.30,0.807143,Good,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP
S02061500_20140923,S02061500,2014-09-23,12.80,0.557143,Moderate,995317.0,6794553.0,Rhin-Meuse,VOSGES,TP


# Interpolate missing values for a site

In [14]:
nans = 0

for site_key in sites:
    site = sites[site_key]
    site = interpolate(site)
    nan_status = site['IBD_EQR_Status'].isna().sum()
    site = safe_fill(site)
    filled_nan_status = site['IBD_EQR_Status'].isna().sum()
    nans += nan_status - filled_nan_status
    sites[site_key] = site

print(f"Total NaNs filled in IBD_EQR_Status: {nans}")

C:\Users\herie\AppData\Local\Temp\ipykernel_54448\4069221416.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev, nxt = site[col].ffill(), site[col].bfill()
C:\Users\herie\AppData\Local\Temp\ipykernel_54448\4069221416.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev, nxt = site[col].ffill(), site[col].bfill()
C:\Users\herie\AppData\Local\Temp\ipykernel_54448\4069221416.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instea

Total NaNs filled in IBD_EQR_Status: 2467


In [15]:
# concat all sites back to a single dataframe
df_filled = pd.concat(sites.values())
df_filled

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02000008_20170703,S02000008,2017-07-03,9.60,0.502924,Poor,1039055.0,6721258.0,Rhin-Meuse,ALSACE,TP
S02000008_20200708,S02000008,2020-07-08,8.00,0.409357,Poor,1039055.0,6721258.0,Rhin-Meuse,ALSACE,TP
S02000010_20070906,S02000010,2007-09-06,14.30,0.777778,Moderate,1039381.0,6737723.0,Rhin-Meuse,ALSACE,None
S02000010_20080811,S02000010,2008-08-11,14.65,0.798246,NaN,1039381.0,6737723.0,Rhin-Meuse,ALSACE,None
S02000010_20090721,S02000010,2009-07-21,15.00,0.818713,Good,1039381.0,6737723.0,Rhin-Meuse,ALSACE,None
...,...,...,...,...,...,...,...,...,...,...
S06999141_20160928,S06999141,2016-09-28,15.60,0.706667,Moderate,939726.0,6583451.0,Rhône-Méditerranée,JURA-PREALPES DU NORD,TP
S06999179_20230628,S06999179,2023-06-28,20.00,1.000000,High,935630.0,6584385.0,Rhône-Méditerranée,JURA-PREALPES DU NORD,TP
S06999180_20160928,S06999180,2016-09-28,19.50,0.966667,High,934863.0,6578148.0,Rhône-Méditerranée,JURA-PREALPES DU NORD,TP


In [16]:
get_nans(df_filled)

Total NaNs in IBD: 1084
Total NaNs in IBD_EQR: 1084
Total NaNs in IBD_EQR_Status: 3196


In [ ]:
# view nans in df_filled
df_filled[df_filled['IBD_EQR_Status'].isna()]

# Get inside ranges

In [17]:
ranges = pd.read_csv("../../data/processed/IBD_EQR_Status_ranges_by_HERlvl1Name.csv")
ranges

,HERlvl1Name,IBD_EQR_Status,IBD_min,IBD_max,IBD_mid
0,ARMORICAIN,Bad,4.5,5.8,5.15
1,ARMORICAIN,Poor,6.3,10.0,8.15
2,ARMORICAIN,Moderate,10.1,13.7,11.90
3,ARMORICAIN,Good,13.8,16.4,15.10
4,ARMORICAIN,High,16.5,20.0,18.25
...,...,...,...,...,...
101,DEPOTS ARGILO SABLEUX,High,17.1,20.0,18.55
102,GRANDS CAUSSES,Poor,9.6,9.6,9.60
103,GRANDS CAUSSES,Moderate,11.3,14.3,12.80
104,GRANDS CAUSSES,Good,14.4,17.0,15.70


In [18]:
# if ibd in ranges, fill ibd_eqr_status accordingly
for site_key in sites:
    site = sites[site_key]
    herlvl1 = site['HERlvl1Name'].values[0]
    site_ranges = ranges[ranges['HERlvl1Name'] == herlvl1]
    for idx, row in site.iterrows():
        ibd_value = row['IBD']
        if pd.notna(ibd_value):
            matching_range = site_ranges[(site_ranges['IBD_min'] <= ibd_value) & (site_ranges['IBD_max'] >= ibd_value)]
            if not matching_range.empty:
                status = matching_range['IBD_EQR_Status'].values[0]
                df_filled.at[idx, 'IBD_EQR_Status'] = status

In [19]:
get_nans(df_filled)

Total NaNs in IBD: 1084
Total NaNs in IBD_EQR: 1084
Total NaNs in IBD_EQR_Status: 1151


In [20]:
# view nans in df_filled
df_filled[df_filled['IBD_EQR_Status'].isna()]

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02001016_20080811,S02001016,2008-08-11,NaN,NaN,NaN,1032286.0,6737746.0,Rhin-Meuse,ALSACE,TP
S02001046_20100726,S02001046,2010-07-26,NaN,NaN,NaN,1045006.0,6801458.0,Rhin-Meuse,ALSACE,TP
S02001500_20070907,S02001500,2007-09-07,NaN,NaN,NaN,1046252.0,6813302.0,Rhin-Meuse,ALSACE,TP
S02001500_20080805,S02001500,2008-08-05,NaN,NaN,NaN,1046252.0,6813302.0,Rhin-Meuse,ALSACE,TP
S02001895_20160721,S02001895,2016-07-21,NaN,NaN,NaN,1023024.0,6730009.0,Rhin-Meuse,ALSACE,TP
...,...,...,...,...,...,...,...,...,...,...
S06830193_20210729,S06830193,2021-07-29,NaN,NaN,NaN,934066.0,6547067.0,Rhône-Méditerranée,JURA-PREALPES DU NORD,TP
S06830239_20190701,S06830239,2019-07-01,NaN,NaN,NaN,958226.0,6581199.0,Rhône-Méditerranée,JURA-PREALPES DU NORD,TP
S06830246_20150604,S06830246,2015-06-04,NaN,NaN,NaN,958147.0,6587830.0,Rhône-Méditerranée,JURA-PREALPES DU NORD,TP


# 1084 codesite sueltos

In [21]:
sites.keys()

dict_keys(['S02000008', 'S02000010', 'S02000011', 'S02000990', 'S02001000', 'S02001006', 'S02001016', 'S02001025', 'S02001030', 'S02001046', 'S02001050', 'S02001100', 'S02001150', 'S02001200', 'S02001250', 'S02001500', 'S02001700', 'S02001715', 'S02001717', 'S02001725', 'S02001738', 'S02001750', 'S02001895', 'S02001915', 'S02001955', 'S02001990', 'S02002000', 'S02002800', 'S02003100', 'S02003200', 'S02003350', 'S02003397', 'S02003400', 'S02003610', 'S02003617', 'S02003620', 'S02003670', 'S02003700', 'S02003800', 'S02003900', 'S02004000', 'S02004300', 'S02004800', 'S02005000', 'S02005688', 'S02005700', 'S02006450', 'S02006500', 'S02007000', 'S02007250', 'S02007300', 'S02007380', 'S02008000', 'S02009000', 'S02009500', 'S02010000', 'S02011000', 'S02013000', 'S02015000', 'S02016050', 'S02017000', 'S02017500', 'S02017850', 'S02018000', 'S02018500', 'S02018690', 'S02018780', 'S02019000', 'S02019200', 'S02019500', 'S02019950', 'S02020000', 'S02020300', 'S02020750', 'S02021000', 'S02021700', '

In [22]:
# create a distance matrix between group of sites
def create_distance_matrix(sites):
    site_keys = list(sites.keys())
    coords = np.array([[sites[key]['Longitude_Lambert93'].values[0], sites[key]['Latitude_Lambert93'].values[0]] for key in site_keys])
    dist_matrix = distance.cdist(coords, coords, metric='euclidean')
    return pd.DataFrame(dist_matrix, index=site_keys, columns=site_keys)

distance_matrix = create_distance_matrix(sites)
distance_matrix

,S02000008,S02000010,S02000011,S02000990,S02001000,S02001006,S02001016,S02001025,S02001030,S02001046,S02001050,S02001100,S02001150,S02001200,S02001250,S02001500,S02001700,S02001715,S02001717,S02001725,S02001738,S02001750,S02001895,S02001915,S02001955,S02001990,S02002000,S02002800,S02003100,S02003200,S02003350,S02003397,S02003400,S02003610,S02003617,S02003620,S02003670,S02003700,S02003800,S02003900,S02004000,S02004300,S02004800,S02005000,S02005688,S02005700,S02006450,S02006500,S02007000,S02007250,S02007300,S02007380,S02008000,S02009000,S02009500,S02010000,S02011000,S02013000,S02015000,S02016050,S02017000,S02017500,S02017850,S02018000,S02018500,S02018690,S02018780,S02019000,S02019200,S02019500,S02019950,S02020000,S02020300,S02020750,S02021000,S02021700,S02022000,S02022465,S02022650,S02022670,S02022675,S02022700,S02022730,S02022760,S02022800,S02023000,S02023500,S02023750,S02024000,S02024300,S02025100,S02025125,S02025150,S02025200,S02025500,S02025700,S02026125,S02026200,S02026250,S02026500,S02026750,S02027000,S02028000,S02028100,S02028300,S02028500,S02029000,S02029160,S02030200,S02030310,S02030350,S02030400,S02030450,S02030500,S02031400,S02031520,S02031650,S02032000,S02032050,S02032400,S02032800,S02035000,S02035600,S02035750,S02036000,S02036250,S02036265,S02037000,S02037400,S02037500,S02038000,S02040800,S02041000,S02041100,S02041230,S02041290,S02041500,S02041575,S02041650,S02041741,S02041750,S02041950,S02042000,S02042340,S02042348,S02042500,S02042520,S02042555,S02042650,S02042700,S02042900,S02043000,S02043050,S02043260,S02043280,S02043300,S02043450,S02043500,S02043600,S02043655,S02043680,S02043690,S02043700,S02043750,S02043775,S02043780,S02043785,S02043790,S02043800,S02043860,S02044000,S02044020,S02044100,S02044300,S02044400,S02045000,S02045050,S02045160,S02045170,S02045200,S02045250,S02045283,S02045350,S02045500,S02046000,S02046400,S02046860,S02047000,S02048960,S02048980,S02049000,S02049120,S02049500,S02049900,S02050000,S02050260,S02051000,S02051500,S02051600,S02051820,S02052000,S02052500,S02052600,S02052800,S02053800,S02054000,S02054100,S02054150,S02054300,S02054500,S02054550,S02054900,S02055000,S02055100,S02055200,S02055500,S02055580,S02056020,S02056200,S02057000,S02057065,S02057168,S02057210,S02057245,S02057250,S02057300,S02057350,S02057400,S02057453,S02057480,S02057490,S02057503,S02057520,S02057525,S02057557,S02057600,S02057650,S02057760,S02058000,S02058990,S02059070,S02059500,S02059760,S02059800,S02060100,S02060200,S02060500,S02060650,S02060700,S02060750,S02061250,S02061500,S02061970,S02063000,S02064000,S02064530,S02064770,S02065090,S02065280,S02065500,S02066000,S02067000,S02067150,S02067153,S02067400,S02067450,S02067505,S02067515,S02067585,S02067600,S02067800,S02067900,S02068500,S02068510,S02068600,S02068700,S02068790,S02068800,S02069100,S02070000,S02070250,S02070300,S02070350,S02070370,S02070500,S02070750,S02070775,S02070810,S02070900,S02071050,S02072050,S02072150,S02072400,S02072600,S02072670,S02072700,S02073200,S02073700,S02074000,S02074900,S02075300,S02076100,S02076180,S02076195,S02076580,S02076800,S02077150,S02077200,S02077300,S02077445,S02077450,S02077700,S02077775,S02078000,S02078500,S02078900,S02079000,S02079250,S02079500,S02080200,S02080500,S02081000,S02081030,S02081100,S02081135,S02081300,S02081600,S02081620,S02082000,S02082160,S02082330,S02082350,S02082710,S02082845,S02082878,S02082900,S02082920,S02082970,S02082985,S02082990,S02084000,S02084100,S02084200,S02084650,S02084800,S02084900,S02085325,S02085600,S02085615,S02085700,S02085850,S02085920,S02086030,S02086170,S02086220,S02086250,S02086300,S02086360,S02086400,S02086450,S02086480,S02086500,S02086550,S02086900,S02088000,S02088400,S02089000,S02089900,S02090000,S02091500,S02092000,S02092600,S02093100,S02093170,S02093200,S02093250,S02093600,S02094000,S02094500,S02094700,S02094800,S02094900,S02094920,S02094940,S02094950,S02094970,S02094973,S02094978,S02094979,S02095250,S02095500,S02095600,S02095900,S02096000,S02096015,S02096480,S02096500,S02096520,S02096575,S02096750

In [24]:
# get nearest sites for a given site
def find_closest_sites(site, df, n=3):
    site_key = site['CodeSite_SamplingOperations_x'].values[0] # this is the site key
    distances = distance_matrix[site_key].sort_values()
    closest_sites_keys = distances.index[1:n+1]  # exclude self
    close = pd.DataFrame()
    for close_key in closest_sites_keys:
        close_site = df[df['CodeSite_SamplingOperations_x'] == close_key]
        close = pd.concat([close, close_site])
        # sort close by Date_SamplingOperation
        close = close.sort_values(by='Date_SamplingOperation')
    return close

In [28]:
site = sites['S02001016']
site

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02001016_20080811,S02001016,2008-08-11,NaN,NaN,NaN,1032286.0,6737746.0,Rhin-Meuse,ALSACE,TP
S02001016_20150718,S02001016,2015-07-18,15.1,0.824561,Good,1032286.0,6737746.0,Rhin-Meuse,ALSACE,TP
S02001016_20170703,S02001016,2017-07-03,14.8,0.807018,Good,1032286.0,6737746.0,Rhin-Meuse,ALSACE,TP
S02001016_20200707,S02001016,2020-07-07,14.8,0.807018,NaN,1032286.0,6737746.0,Rhin-Meuse,ALSACE,TP


In [29]:
# create a dictionary of dataframes per site and later access them by 1, 2, ...
sites = {}
for site, group in df_filled.groupby('CodeSite_SamplingOperations_x'):
    sites[site] = group

In [30]:
site = sites['S02001016']
site

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02001016_20080811,S02001016,2008-08-11,NaN,NaN,NaN,1032286.0,6737746.0,Rhin-Meuse,ALSACE,TP
S02001016_20150718,S02001016,2015-07-18,15.1,0.824561,Good,1032286.0,6737746.0,Rhin-Meuse,ALSACE,TP
S02001016_20170703,S02001016,2017-07-03,14.8,0.807018,Good,1032286.0,6737746.0,Rhin-Meuse,ALSACE,TP
S02001016_20200707,S02001016,2020-07-07,14.8,0.807018,Good,1032286.0,6737746.0,Rhin-Meuse,ALSACE,TP


In [26]:
closest = find_closest_sites(site, df_filled, n=3)
closest

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02001025_20070905,S02001025,2007-09-05,13.10,0.707602,Moderate,1028974.0,6740387.0,Rhin-Meuse,ALSACE,TP
S02001000_20070906,S02001000,2007-09-06,14.40,0.783626,Good,1038791.0,6736177.0,Rhin-Meuse,ALSACE,TP
S02001025_20080811,S02001025,2008-08-11,14.70,0.801170,Good,1028974.0,6740387.0,Rhin-Meuse,ALSACE,TP
S02001000_20080811,S02001000,2008-08-11,14.70,0.801170,Good,1038791.0,6736177.0,Rhin-Meuse,ALSACE,TP
S02001025_20090721,S02001025,2009-07-21,13.50,0.730994,Moderate,1028974.0,6740387.0,Rhin-Meuse,ALSACE,TP
S02001000_20090721,S02001000,2009-07-21,14.90,0.812865,Good,1038791.0,6736177.0,Rhin-Meuse,ALSACE,TP
S02001000_20100719,S02001000,2010-07-19,15.20,0.830409,Good,1038791.0,6736177.0,Rhin-Meuse,ALSACE,TP
S02001025_20100719,S02001025,2010-07-19,14.30,0.777778,Moderate,1028974.0,6740387.0,Rhin-Meuse,ALSACE,TP
S02001000_20110723,S02001000,2011-07-23,15.70,0.859649,Good,1038791.0,6736177.0,Rhin-Meuse,ALSACE,TP


In [31]:
site = sites['S02001046']
site

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02001046_20100726,S02001046,2010-07-26,NaN,NaN,NaN,1045006.0,6801458.0,Rhin-Meuse,ALSACE,TP


In [32]:
closest = find_closest_sites(site, df_filled, n=3)
closest

,CodeSite_SamplingOperations_x,Date_SamplingOperation,IBD,IBD_EQR,IBD_EQR_Status,Longitude_Lambert93,Latitude_Lambert93,Watershed,HERlvl1Name,Streamsize
SamplingOperations_code,,,,,,,,,,
S02001725_20070907,S02001725,2007-09-07,13.8,0.748538,Moderate,1044874.0,6806422.0,Rhin-Meuse,ALSACE,TP
S02001715_20070907,S02001715,2007-09-07,14.3,0.777778,Moderate,1037861.0,6797189.0,Rhin-Meuse,ALSACE,None
S02001715_20080806,S02001715,2008-08-06,16.0,0.877193,Good,1037861.0,6797189.0,Rhin-Meuse,ALSACE,None
S02001725_20080912,S02001725,2008-09-12,14.5,0.789474,Good,1044874.0,6806422.0,Rhin-Meuse,ALSACE,TP
S02001715_20090723,S02001715,2009-07-23,17.1,0.941520,High,1037861.0,6797189.0,Rhin-Meuse,ALSACE,None
S02001725_20090804,S02001725,2009-08-04,14.7,0.801170,Good,1044874.0,6806422.0,Rhin-Meuse,ALSACE,TP
S02001715_20100722,S02001715,2010-07-22,16.5,0.906433,Good,1037861.0,6797189.0,Rhin-Meuse,ALSACE,None
S02001725_20100726,S02001725,2010-07-26,14.6,0.795322,Good,1044874.0,6806422.0,Rhin-Meuse,ALSACE,TP
S02001725_20110721,S02001725,2011-07-21,14.6,0.795322,Good,1044874.0,6806422.0,Rhin-Meuse,ALSACE,TP
